# NB03 · Feature Engineering, Correlation & Statistical Feature Selection

In [0]:
%run ./NB01_Config_and_Setup

## 1 · Load & Label Billings (oldest Co_Ref + reference_date)

In [0]:
section("Loading tables")
billings      = load_table(BILLINGS_TABLE,      BILLINGS_TABLE)
cc_calls      = load_table(CC_CALLS_TABLE,      CC_CALLS_TABLE)
emails        = load_table(EMAILS_TABLE,        EMAILS_TABLE)
renewal_calls = load_table(RENEWAL_CALLS_TABLE, RENEWAL_CALLS_TABLE)


In [0]:
section("Billings — deduplication: keep oldest Co_Ref")
billings = parse_dates(billings, ['Prospect_Renewal_Date', 'Closed_Date'])
before = len(billings)
billings = (billings
            .sort_values('Prospect_Renewal_Date', ascending=True)
            .drop_duplicates(subset='Co_Ref', keep='first'))
print(f"  {before:,} rows → {len(billings):,} unique Co_Ref "
      f"({before-len(billings):,} non-oldest records removed)")


In [0]:
section("Billings — compute reference_date")
billings['days_to_close'] = (
    billings['Closed_Date'] - billings['Prospect_Renewal_Date']).dt.days

billings['reference_date'] = billings['Closed_Date'].fillna(
    billings['Prospect_Renewal_Date'] + pd.Timedelta(days=CHURN_DAYS_THRESHOLD))

print(f"  Using Closed_Date      : {billings['Closed_Date'].notna().sum():,} customers")
print(f"  Using PRD + {CHURN_DAYS_THRESHOLD} days     : {billings['Closed_Date'].isna().sum():,} customers")


In [0]:
section("Billings — apply churn labels")
conditions = [
    (billings['Prospect_Outcome'] == 'Churned') & (billings['Closed_Date'].isna()),
    (billings['Prospect_Outcome'] == 'Churned') & (billings['days_to_close'] > CHURN_DAYS_THRESHOLD),
    (billings['Prospect_Outcome'] == 'Churned') & (billings['days_to_close'] < 0),
    billings['Prospect_Outcome'] == 'Won',
    billings['Prospect_Outcome'] == 'Open',
]
billings['Churn_Label'] = np.select(conditions,
    ['Churned', 'Churned', 'Pre_Churned', 'Won', 'Open'], default='Churned')
model_df = billings[billings['Churn_Label'].isin(['Won', 'Churned'])].copy()
model_df['Target']       = (model_df['Churn_Label'] == 'Churned').astype(int)
model_df['Renewal_Year'] = model_df['Prospect_Renewal_Date'].dt.year
print(f"model_df: {len(model_df):,} rows  |  churn rate: {model_df['Target'].mean()*100:.2f}%")


## 2 · CC Calls — Nearest Call to reference_date

In [0]:
section("CC Calls — select call nearest to reference_date per customer")
cc_calls = parse_dates(cc_calls, ['Call_Date'])

cc_with_ref = cc_calls.merge(
    model_df[['Co_Ref', 'reference_date']], on='Co_Ref', how='inner')
cc_with_ref['days_from_ref'] = (
    cc_with_ref['Call_Date'] - cc_with_ref['reference_date']).dt.days.abs()

cc_nearest = (cc_with_ref
              .sort_values(['Co_Ref', 'days_from_ref'])
              .drop_duplicates('Co_Ref', keep='first')
              .copy())

print(f"CC Calls: {len(cc_calls):,} total rows → {len(cc_nearest):,} unique customers selected")
print(f"  Median days between selected call and reference_date : "
      f"{cc_nearest['days_from_ref'].median():.0f}")
print(f"  % calls within 90 days of reference_date            : "
      f"{(cc_nearest['days_from_ref']<=90).mean()*100:.1f}%")


In [0]:
section("CC Calls — binary flag encoding (screened columns only)")
CC_BIN_FINAL = [
    'cc_customer_issues_concerns', 'cc_business_struggles_financial_hardship',
    'cc_dissatisfaction_support', 'cc_pricing_mentioned', 'cc_refund_discussed',
    'cc_contractor_suggest_leave', 'cc_contractor_complained',
]
cc_nearest = binary_flags(cc_nearest, CC_BIN_FINAL)
print("Binary flags encoded:", CC_BIN_FINAL)


In [0]:
section("CC Calls — sentiment score encoding and imputation")
CC_NUM_FINAL = [
    'cc_contractor_sentiment_start_score',
    'cc_contractor_sentiment_end_score',
    'cc_contractor_sentiment_overall_score',
]
for c in CC_NUM_FINAL:
    cc_nearest[c] = pd.to_numeric(cc_nearest[c], errors='coerce')
    null_pct   = cc_nearest[c].isna().mean() * 100
    median_val = cc_nearest[c].median()
    cc_nearest[c] = cc_nearest[c].fillna(median_val)
    print(f"  {c}: null%={null_pct:.1f}% → imputed with median={median_val:.1f}")


In [0]:
section("CC Calls — feature frame")
CC_ALL_COLS = CC_BIN_FINAL + CC_NUM_FINAL
cc_feat = cc_nearest[['Co_Ref'] + CC_ALL_COLS].drop_duplicates('Co_Ref').copy()
print(f"CC feature frame: {cc_feat.shape}")
display_df(cc_feat.head(5))


## 3 · Emails — Nearest TTR Bucket

In [0]:
section("Emails — select nearest TTR bucket per customer")
TTR_ORDER = {'pre_renewal': 1, '14_out': 2, '45_out': 3, 'prior_year': 4}
emails['_ttr_rank'] = emails['Time_to_Renewal'].map(TTR_ORDER).fillna(5)
em_nearest = (emails.sort_values(['Co_Ref', '_ttr_rank'])
                    .drop_duplicates('Co_Ref', keep='first')
                    .copy())

print(f"Emails: {len(emails):,} total rows → {len(em_nearest):,} unique customers selected")
print("\nBucket distribution of selected emails:")
print(em_nearest['Time_to_Renewal'].value_counts().to_string())
print()
print("NOTE: Customers with only prior_year emails have no closer record available.")
print("      prior_year is their fallback — it still carries pre-churn signal,")
print("      just from the preceding renewal cycle.")


In [0]:
section("Emails — binary flag encoding")
EM_BIN_FINAL = [
    'crm_delays_in_accreditation', 'crm_contractor_suggested_leave',
    'crm_competitors_mentioned', 'crm_platform_issues_raised',
    'crm_customer_complained', 'crm_refund_mentioned',
    'crm_negative_customer_experience', 'crm_dissatisfaction_with_support',
    'crm_financial_hardship_mentioned', 'crm_agent_chased_contractor',
    'crm_dissatisified_with_renewal_price',
]
em_nearest = binary_flags(em_nearest, EM_BIN_FINAL)
print("Binary flags encoded:", EM_BIN_FINAL)


In [0]:
section("Emails — numeric encoding and null imputation")
em_nearest['crm_agent_chase_count'] = pd.to_numeric(
    em_nearest['crm_agent_chase_count'], errors='coerce')
chase_median = em_nearest['crm_agent_chase_count'].median()
em_nearest['crm_agent_chase_count'] = em_nearest['crm_agent_chase_count'].fillna(chase_median)
print(f"crm_agent_chase_count: null imputed with median={chase_median:.1f}")

em_nearest['crm_contractor_sentiment_score'] = pd.to_numeric(
    em_nearest['crm_contractor_sentiment_score'], errors='coerce')
sentiment_median = em_nearest['crm_contractor_sentiment_score'].median()
em_nearest['crm_sentiment_missing'] = em_nearest['crm_contractor_sentiment_score'].isna().astype(int)
em_nearest['crm_contractor_sentiment_score'] = em_nearest['crm_contractor_sentiment_score'].fillna(sentiment_median)
print(f"crm_contractor_sentiment_score: null imputed with median={sentiment_median:.1f} "
      f"+ _missing flag added")


In [0]:
section("Emails — feature frame")
EM_ALL_COLS = EM_BIN_FINAL + ['crm_agent_chase_count',
                               'crm_contractor_sentiment_score',
                               'crm_sentiment_missing']
em_feat = em_nearest[['Co_Ref'] + EM_ALL_COLS].drop_duplicates('Co_Ref').copy()
print(f"Email feature frame: {em_feat.shape}")
display_df(em_feat.head(5))


## 4 · Renewal Calls — Nearest Call to reference_date

In [0]:
section("Renewal Calls — select call nearest to reference_date per customer")
renewal_calls = parse_dates(renewal_calls, ['Call_Date'])

rc_with_ref = renewal_calls.merge(
    model_df[['Co_Ref', 'reference_date']], on='Co_Ref', how='inner')
rc_with_ref['days_from_ref'] = (
    rc_with_ref['Call_Date'] - rc_with_ref['reference_date']).dt.days.abs()

rc_nearest = (rc_with_ref
              .sort_values(['Co_Ref', 'days_from_ref'])
              .drop_duplicates('Co_Ref', keep='first')
              .copy())

print(f"Renewal Calls: {len(renewal_calls):,} total rows → "
      f"{len(rc_nearest):,} unique customers selected")
print(f"  Median days between selected call and reference_date : "
      f"{rc_nearest['days_from_ref'].median():.0f}")
print(f"  % calls within 90 days of reference_date            : "
      f"{(rc_nearest['days_from_ref']<=90).mean()*100:.1f}%")


In [0]:
section("Renewal Calls — binary flag encoding and imputation")
RC_BIN_FINAL = [
    'Serious_Complaint', 'Other_Complaint', 'Discussion_on_Price_Increase',
    'Explicit_Competitor_Mention', 'Explicit_Switching_Intent', 'Desire_To_Cancel',
    'Discount_Offered', 'Call_Reschedule_Request',
    'Agent_Flagged_Membership_Status_Alert',
    'Renewal_Impact_Due_to_Price_Increase',
]
RC_BIN_AVAILABLE = [c for c in RC_BIN_FINAL if c in rc_nearest.columns]
for c in RC_BIN_AVAILABLE:
    rc_nearest[c] = rc_nearest[c].isin(['Yes', '1', 1, True]).astype(int)
    null_pct = rc_nearest[c].isna().mean() * 100
    if null_pct > 0:
        rc_nearest[c] = rc_nearest[c].fillna(0)
        print(f"  {c}: null%={null_pct:.1f}% → imputed with 0")
print("Binary flags encoded for Renewal Calls.")


In [0]:
section("Renewal Calls — feature frame")
rc_feat = rc_nearest[['Co_Ref'] + RC_BIN_AVAILABLE].drop_duplicates('Co_Ref').copy()
print(f"Renewal Calls feature frame: {rc_feat.shape}")
display_df(rc_feat.head(5))


## 5 · Billings — Feature Building

In [0]:
section("Billings — numeric feature encoding and imputation")
BILL_NUM_FINAL = [
    'Auto_Renewal_Score', 'Status_Scores', 'Anchoring_Score', 'Tenure_Scores',
    'Tenure_Years', 'Current_Anchorings', 'Last_Years_Price', 'Starting_Net',
    'Total_Renewal_Score_New', 'Sustainability_Score', 'Renewal_Score_At_Release',
    'Proforma_Approved_Lists',
]
for c in BILL_NUM_FINAL:
    if c not in model_df.columns: continue
    model_df[c] = pd.to_numeric(model_df[c], errors='coerce')
    null_pct   = model_df[c].isna().mean() * 100
    median_val = model_df[c].median()
    model_df[c] = model_df[c].fillna(median_val)
    print(f"  {c}: null%={null_pct:.1f}% → imputed with median={median_val:.1f}")


In [0]:
section("Billings — categorical encoding")
BAND_ORDER = ['Band A','Band B','Band C1','Band C2','Band D','Band E',
              'Band F','Band F1','Band F2','Band G','Band H','Band I','Band J','Group']
model_df['Band_Code'] = pd.Categorical(
    model_df['Band'], categories=BAND_ORDER, ordered=True).codes

if 'Payment_Method' in model_df.columns:
    model_df['is_card_payment'] = (model_df['Payment_Method'] == 'CARD').astype(int)

print("Band_Code encoded (Band A=0 → Group=13)")
print("is_card_payment encoded (CARD=1, other=0)")


## 6 · Engineered Composite Features

In [0]:
section("Composite Feature 1 — Price Pressure Index")
from scipy.stats import pointbiserialr
model_df['price_pressure_index'] = (
    (model_df['Starting_Net'] - model_df['Last_Years_Price']) /
    (model_df['Last_Years_Price'] + 1)
).clip(-2, 5)
r, p = pointbiserialr(model_df['Target'], model_df['price_pressure_index'].fillna(0))
print(f"price_pressure_index: r={r:+.4f}, p={p:.3e}")
print("  Formula : (Starting_Net - Last_Years_Price) / (Last_Years_Price + 1)")
print("  Rationale: both Starting_Net (r=-0.092) and Last_Years_Price (r=-0.117) are")
print("             significant individually; their ratio captures the customer's")
print("             perceived price burden change — positive = price went up.")


In [0]:
section("Composite Feature 2 — Zero Anchor Flag")
model_df['zero_anchor_flag'] = (model_df['Current_Anchorings'] == 0).astype(int)
r, p = pointbiserialr(model_df['Target'], model_df['zero_anchor_flag'])
print(f"zero_anchor_flag: r={r:+.4f}, p={p:.3e}")
print("  Rationale: Current_Anchorings (r=-0.093) — zero anchors = no structural")
print("             lock-in. Binary flag captures this threshold effect cleanly.")


In [0]:
section("Composite Feature 3 — Renewal Health Score Decline")
model_df['score_decline'] = (
    model_df['Renewal_Score_At_Release'] - model_df['Total_Renewal_Score_New']
).clip(-50, 50)
r, p = pointbiserialr(model_df['Target'], model_df['score_decline'].fillna(0))
print(f"score_decline: r={r:+.4f}, p={p:.3e}")
print("  Rationale: Total_Renewal_Score_New (r=-0.636) and")
print("             Renewal_Score_At_Release (r=-0.289) are both significant.")
print("             Their difference captures how much health deteriorated")
print("             between release and final scoring.")


## 7 · Merge All Feature Frames

In [0]:
section("Merging all feature frames into final dataset")
ENGN_COLS = ['price_pressure_index', 'zero_anchor_flag', 'score_decline']
BILL_COLS = BILL_NUM_FINAL + ['Band_Code', 'is_card_payment']

merged = model_df.copy()
row_before = len(merged)

merged = merged.merge(cc_feat, on='Co_Ref', how='left')
print(f"  After CC merge  : {len(merged):,} rows | "
      f"customers with CC data: {merged[cc_feat.columns[1]].notna().sum():,}")

merged = merged.merge(em_feat, on='Co_Ref', how='left')
print(f"  After Email merge: {len(merged):,} rows | "
      f"customers with email data: {merged[em_feat.columns[1]].notna().sum():,}")

merged = merged.merge(rc_feat, on='Co_Ref', how='left')
print(f"  After RC merge  : {len(merged):,} rows | "
      f"customers with RC data: {merged[rc_feat.columns[1]].notna().sum():,}")

assert len(merged) == row_before, "Row count changed — check for Co_Ref duplicates in feature frames"
print(f"  Row count preserved: {len(merged):,}  ✓")

CC_COLS = [c for c in cc_feat.columns if c != 'Co_Ref']
EM_COLS = [c for c in em_feat.columns if c != 'Co_Ref']
RC_COLS = [c for c in rc_feat.columns if c != 'Co_Ref']
merged[CC_COLS + EM_COLS + RC_COLS] = merged[CC_COLS + EM_COLS + RC_COLS].fillna(0)

FINAL_FEATURES = CC_COLS + EM_COLS + RC_COLS + BILL_COLS + ENGN_COLS
FINAL_FEATURES = [f for f in FINAL_FEATURES if f in merged.columns]
print(f"\nTotal candidate features: {len(FINAL_FEATURES)}")


## 8 · Correlation — Feature vs Target

In [0]:
section("Point-Biserial Correlation — All Features vs Churn Target")
corr_rows = []
for feat in FINAL_FEATURES:
    col = pd.to_numeric(merged[feat], errors='coerce').fillna(0)
    r, p = pointbiserialr(merged['Target'], col)
    corr_rows.append({'Feature': feat, 'r': round(r, 4), 'p_value': round(p, 8),
                      'Abs_r': round(abs(r), 4),
                      'Significant': 'Yes' if p < 0.05 else 'No'})
corr_df = pd.DataFrame(corr_rows).sort_values('Abs_r', ascending=False)
display_df(corr_df, "Feature Correlation with Churn Target (sorted by |r|)")
print(f"\nSignificant features (p<0.05): {(corr_df['Significant']=='Yes').sum()} / {len(corr_df)}")


In [0]:
fig, ax = plt.subplots(figsize=(12, max(6, len(FINAL_FEATURES)*0.27)))
top_corr = corr_df.head(30)
colors = [PALETTE['danger'] if v > 0 else PALETTE['green'] for v in top_corr['r']]
ax.barh(range(len(top_corr)), top_corr['r'], color=colors, edgecolor='#0a0e1a')
ax.set_yticks(range(len(top_corr)))
ax.set_yticklabels(top_corr['Feature'], fontsize=8)
ax.set_xlabel("Point-Biserial r  (positive = higher value → more churn)")
ax.set_title("Top 30 Features — Correlation with Churn Target", fontsize=13)
ax.axvline(0, color='white', linewidth=0.5)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=PALETTE['danger'], label='Increases churn risk'),
                   Patch(color=PALETTE['green'],  label='Reduces churn risk')])
plt.tight_layout(); plt.show()


## 9 · Mann-Whitney U — Remove Non-Significant Features

In [0]:
section("Mann-Whitney U Test — Distribution shift: Churned vs Retained")
from scipy.stats import mannwhitneyu

churn_grp    = merged[merged['Target'] == 1]
retained_grp = merged[merged['Target'] == 0]

mw_rows = []
for feat in FINAL_FEATURES:
    a = pd.to_numeric(churn_grp[feat],    errors='coerce').dropna()
    b = pd.to_numeric(retained_grp[feat], errors='coerce').dropna()
    if len(a) < 5 or len(b) < 5: continue
    stat, p = mannwhitneyu(a, b, alternative='two-sided')
    mw_rows.append({'Feature': feat, 'p_value': round(p, 8),
                    'Churn_Mean': round(a.mean(), 4), 'Retained_Mean': round(b.mean(), 4),
                    'Significant': 'Yes' if p < 0.05 else 'No'})
mw_df = pd.DataFrame(mw_rows).sort_values('p_value')
display_df(mw_df, "Mann-Whitney U Results — All Features")

mw_drop = mw_df[mw_df['Significant'] == 'No']['Feature'].tolist()
if mw_drop:
    print(f"\nDropping non-significant features: {mw_drop}")
    FINAL_FEATURES = [f for f in FINAL_FEATURES if f not in mw_drop]

## 10 · VIF — Multicollinearity Check

In [0]:
section("Variance Inflation Factor — Multicollinearity")
from statsmodels.stats.outliers_influence import variance_inflation_factor

feat_matrix = merged[FINAL_FEATURES].apply(pd.to_numeric, errors='coerce').fillna(0)
vif_rows = []
for i, col in enumerate(FINAL_FEATURES):
    try:
        v = variance_inflation_factor(feat_matrix.values, i)
        vif_rows.append({'Feature': col, 'VIF': round(v, 2)})
    except Exception:
        pass

vif_df = pd.DataFrame(vif_rows).sort_values('VIF', ascending=False)
vif_df['Action'] = vif_df['VIF'].apply(
    lambda v: 'DROP (VIF>10 — multicollinear)' if v > 10 else
              'MONITOR (VIF 5–10)'              if v > 5  else 'KEEP (VIF<5)')
display_df(vif_df, "VIF Results")

vif_drop = vif_df[vif_df['VIF'] > 10]['Feature'].tolist()
if vif_drop:
    print(f"\nDropping high-VIF features: {vif_drop}")
    FINAL_FEATURES = [f for f in FINAL_FEATURES if f not in vif_drop]

else:
    print("\nNo features exceed VIF=10 — all retained.")


## 11 · Final Feature Summary

In [0]:
section(f"FINAL FEATURE SET features passed all gates")
final_feat_df = corr_df[corr_df['Feature'].isin(FINAL_FEATURES)].copy()
final_feat_df['Source'] = final_feat_df['Feature'].apply(
    lambda x: 'CC Calls'     if x.startswith('cc_')  else
              'Emails'        if x.startswith('crm_') else
              'Renewal Calls' if x[0].isupper()        else
              'Engineered'    if x in ENGN_COLS         else 'Billing')
display_df(
    final_feat_df[['Feature','Source','r','p_value','Abs_r']].sort_values('Abs_r', ascending=False),
)
